## <b> Построение графа знаний </b>

In [ ]:
# Стандартные библиотеки
import pandas as pd
import urllib.parse
from tqdm import tqdm

# Библиотеки для построения графа
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, XSD

# Библиотеки для визуализации
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network

### Загрузка данных

In [12]:
# Загружаем таблицы
books       = pd.read_csv("all_book.csv")
text_marks  = pd.read_csv("all_mark_text.csv")
image_marks = pd.read_csv("all_mark_image.csv")
audio_marks = pd.read_csv("all_mark_audio.csv")

### Построение и визуализация графа

<img src="structure_graph.png" width="1200">

In [ ]:
g = Graph()
EX = Namespace("http://example.org/schema#")
g.bind("ex", EX)

for _, book in books.iterrows():
    book_id   = str(book["BOOK ID"])
    reader_id = str(book["READER"])

    book_uri    = URIRef(f"http://example.org/book/{book_id}")
    chapter_val = book.get("CH. TITLE", f"Chapter Unknown")
    chapter_uri = URIRef(f"http://example.org/chapter/{book_id}_{urllib.parse.quote(chapter_val)}")

    # Узел книги
    g.add((book_uri, RDF.type, EX.Book))
    
    # Добавим title как литерал
    if "PROJECT TITLE" in book:
        g.add((book_uri, EX.hasTitle, Literal(book["PROJECT TITLE"])))

    # Добавим genre
    if "GENRE" in book:
        g.add((book_uri, EX.hasGenre, Literal(book["GENRE"])))

    # Связь с главой
    g.add((book_uri,    EX.hasChapter,   chapter_uri))
    g.add((chapter_uri, RDF.type,        EX.Chapter))
    g.add((chapter_uri, EX.chapterValue, Literal(chapter_val)))

    # Связь главы с читателем
    reader_uri = URIRef(f"http://example.org/reader/{reader_id}")
    g.add((chapter_uri, EX.hasReader, reader_uri))
    g.add((reader_uri,  RDF.type,     EX.Reader))
    g.add((reader_uri,  EX.readerID,  Literal(reader_id)))

    # Разметка изображений (привязана к книге напрямую)
    imgs = image_marks[image_marks["BOOK ID"] == int(book_id)]
    for _, row in imgs.iterrows():
        filename      = str(row["image"]).replace("\\", "/")
        safe_filename = urllib.parse.quote(filename, safe="")
        img_uri       = URIRef(f"http://example.org/imagemark/{book_id}_{safe_filename}")

        g.add((img_uri, RDF.type, EX.ImageMark))
        g.add((book_uri, EX.hasMarkImage, img_uri))

        for col in ["scene", "characters", "style", "attributes"]:
            if pd.notna(row[col]):
                g.add((img_uri, URIRef(EX + f"has{col.capitalize()}"), Literal(row[col])))
        g.add((img_uri, EX.filename, Literal(filename)))

    # Разметка текста (от reader)
    text_rows = text_marks[text_marks["BOOK ID"] == int(book_id)]
    for _, row in text_rows.iterrows():
        text_uri = URIRef(f"http://example.org/textmark/{book_id}_{row['token_id']}")
        g.add((text_uri, RDF.type, EX.TextMark))
        g.add((reader_uri, EX.hasMarkText, text_uri))
        g.add((text_uri,   EX.hasText,     Literal(row["text"])))
        g.add((text_uri,   EX.hasPosText,  Literal(row["POS_tags"])))
        g.add((text_uri,   EX.hasNERText,  Literal(row["NER_tags"])))
        g.add((text_uri,   EX.hasLabel,    Literal(row["overall_mood"])))

    # Разметка аудио (от reader)
    audio_rows = audio_marks[audio_marks["BOOK ID"] == int(book_id)]
    for _, row in audio_rows.iterrows():
        audio_uri = URIRef(f"http://example.org/audiomark/{book_id}_{row['token_id']}")
        g.add((audio_uri, RDF.type, EX.AudioMark))
        g.add((reader_uri, EX.hasMarkAudio, audio_uri))
        g.add((audio_uri,  EX.hasLabel,     Literal(row["label"])))
        g.add((audio_uri,  EX.during,       Literal(row["end"], datatype=XSD.float)))
        g.add((audio_uri,  EX.temp,         Literal(row["temp"])))
        g.add((audio_uri,  EX.gender,       Literal(row["speaker's gender"])))
        g.add((audio_uri,  EX.source_file,  Literal(row["source_file"])))

# Сохраняем граф
g.serialize("knowledge_graph.ttl", format="turtle")

<Graph identifier=Na9cc85627d6a4371b68ff6ba7b6304b4 (<class 'rdflib.graph.Graph'>)>

In [ ]:
# Загрузка RDF-графа
g = Graph()
g.parse("knowledge_graph.ttl", format="turtle")

# Инициализация интерактивного графа
net = Network(height="750px", width="100%", directed=True, notebook=False)
net.force_atlas_2based()  # красивое распределение узлов

# Добавим узлы и связи
for s, p, o in tqdm(g):
    subj = str(s)
    pred = str(p).split("#")[-1]
    obj  = str(o)

    # Упрощённые подписи
    subj_label = subj.split("/")[-1][:30]
    obj_label  = obj.split("/")[-1][:30]

    # Добавление узлов
    net.add_node(subj, label = subj_label, title = subj, color = "#C1A2FF")
    net.add_node(obj,  label = obj_label,  title = obj,  color = "#C1A2FF")

    # Добавим ребро с подписью
    net.add_edge(subj, obj, label = pred)

# Сохраняем и открываем в браузере
net.write_html("rdf_graph.html")

100%|██████████| 324931/324931 [18:01<00:00, 300.40it/s]  


In [ ]:
# Загружаем RDF-граф
g = Graph()
g.parse("knowledge_graph.ttl", format="turtle")

# Укажи URI нужной книги
book_id = "60"
book_uri = f"http://example.org/book/{book_id}"

# Создаем интерактивный граф
net = Network(height="1440px", width="100%", directed=True)
net.force_atlas_2based()

# Добавим только триплеты, связанные с книгой
for s, p, o in tqdm(g.triples((URIRef(book_uri), None, None))):
    net.add_node(str(s), label = s.split("/")[-1], color = "#C1A2FF")
    net.add_node(str(o), label = o.split("/")[-1], color = "#C1A2FF")
    net.add_edge(str(s), str(o), label = p.split("#")[-1])

    # Также включим "дочерние" узлы объекта
    for s2, p2, o2 in g.triples((o, None, None)):
        net.add_node(str(o2), label = str(o2).split("/")[-1], color = "#F090D3")
        net.add_edge(str(s2), str(o2), label = p2.split("#")[-1])

        for s3, p3, o3 in g.triples((o2, None, None)):
            net.add_node(str(o3), label = str(o3).split("/")[-1], color = "#A2DAFF")
            net.add_edge(str(s3), str(o3), label = p3.split("#")[-1])

            for s4, p4, o4 in g.triples((o3, None, None)):
                net.add_node(str(o4), label = str(o4).split("/")[-1], color = "#FFE8A2")
                net.add_edge(str(s4), str(o4), label = p4.split("#")[-1])

# Сохраняем HTML
net.write_html("book_subgraph.html")

5it [00:00, 160.97it/s]


<img src="for_1book.png" width="700">

### Запросы к графу

In [86]:
# Загружаем граф
g = Graph()
g.parse("knowledge_graph.ttl", format="turtle")

# Указываем ID читателя
reader_id = "198"

# SPARQL-запрос
query = f"""
PREFIX ex: <http://example.org/schema#>

SELECT DISTINCT ?book_id ?project_title ?chapter_name
WHERE {{
        ?reader a ex:Reader ;
                ex:readerID "{reader_id}" ;
                ^ex:hasReader ?chapter .

        ?chapter a ex:Chapter ;
                ex:chapterValue ?chapter_name ;
                ^ex:hasChapter ?book .

        ?book a ex:Book ;
              ex:hasTitle ?project_title .

        BIND(REPLACE(STR(?book), "^.*/book/", "") AS ?book_id)
      }}
"""

# Выполнение запроса
results = g.query(query)

# Вывод
for row in results:
    print("BOOK ID      :", row.book_id)
    print("PROJECT TITLE:", row.project_title)
    print("CHAPTER      :", row.chapter_name)
    print("-" * 40)

BOOK ID      : 121
PROJECT TITLE: Northanger Abbey
CHAPTER      : Chapter 12
----------------------------------------
BOOK ID      : 161
PROJECT TITLE: Sense and Sensibility
CHAPTER      : Chapter 18
----------------------------------------
BOOK ID      : 5342
PROJECT TITLE: Story Girl
CHAPTER      : 16 - The Ghostly Bell
----------------------------------------


In [89]:
g = Graph()
g.parse("knowledge_graph.ttl", format="turtle")

# ID книги
book_id = "121"

query = f"""
PREFIX ex: <http://example.org/schema#>

SELECT DISTINCT ?chapter ?reader_id ?text ?label
WHERE {{
  ?book a ex:Book .
  FILTER (STRENDS(STR(?book), "/book/{book_id}"))

  ?book ex:hasChapter ?chapter_uri .
  ?chapter_uri ex:chapterValue ?chapter ;
               ex:hasReader ?reader .

  ?reader ex:readerID ?reader_id ;
          ex:hasMarkText ?textmark .

  ?textmark ex:hasText ?text ;
            ex:hasLabel ?label .
}}
"""

results = g.query(query)

for row in results:
    print("CHAPTER     :", row.chapter)
    print("READER ID   :", row.reader_id)
    print("TEXT        :", row.text)
    print("ОVERALL MOOD:", row.label)
    print("-" * 40)

CHAPTER     : Chapter 01
READER ID   : 19
TEXT        : A few days passed away and Catherine though not allowing herself to suspect her friend could not help watching her closely
ОVERALL MOOD: eerie
----------------------------------------
CHAPTER     : Chapter 01
READER ID   : 19
TEXT        : Catherine was completely awakened
ОVERALL MOOD: eerie
----------------------------------------
CHAPTER     : Chapter 01
READER ID   : 19
TEXT        : Chapter Thirty
ОVERALL MOOD: eerie
----------------------------------------
CHAPTER     : Chapter 01
READER ID   : 19
TEXT        : Chapter twenty one
ОVERALL MOOD: eerie
----------------------------------------
CHAPTER     : Chapter 01
READER ID   : 19
TEXT        : Instantaneously with the consciousness of existence returned her recollection of the manuscript
ОVERALL MOOD: eerie
----------------------------------------
CHAPTER     : Chapter 01
READER ID   : 19
TEXT        : Missus Allen said Catherine the next morning Will there be any harm in m

In [91]:
# Загружаем граф
g = Graph()
g.parse("knowledge_graph.ttl", format="turtle")

# Жанр для фильтрации
genre = "Fantasy / Sci-Fi"

# SPARQL-запрос
query = f"""
PREFIX ex: <http://example.org/schema#>

SELECT DISTINCT ?book_id ?project_title ?chapter_name ?reader_id
WHERE {{
        ?book a ex:Book ;
              ex:hasGenre "{genre}" ;
              ex:hasTitle ?project_title ;
              ex:hasChapter ?chapter .

        ?chapter a ex:Chapter ;
                ex:chapterValue ?chapter_name ;
                ex:hasReader ?reader .

        ?reader a ex:Reader ;
                ex:readerID ?reader_id .

        BIND(REPLACE(STR(?book), "^.*/book/", "") AS ?book_id)
      }}
"""

# Выполнение запроса
results = g.query(query)

# Вывод
for row in results:
    print("BOOK ID      :", row.book_id)
    print("PROJECT TITLE:", row.project_title)
    print("CHAPTER      :", row.chapter_name)
    print("READER ID    :", row.reader_id)
    print("-" * 40)

BOOK ID      : 10624
PROJECT TITLE: John Silence
CHAPTER      : A Psychical Invasion part 6
READER ID    : 8108
----------------------------------------
BOOK ID      : 1210
PROJECT TITLE: Kwaidan: Stories and Studies of Strange Things
CHAPTER      : Jikininki
READER ID    : 2952
----------------------------------------
BOOK ID      : 1210
PROJECT TITLE: Kwaidan: Stories and Studies of Strange Things
CHAPTER      : Of A Mirror and a Bell
READER ID    : 2952
----------------------------------------
BOOK ID      : 1210
PROJECT TITLE: Kwaidan: Stories and Studies of Strange Things
CHAPTER      : Rokuro-Kubi
READER ID    : 2952
----------------------------------------
BOOK ID      : 12163
PROJECT TITLE: Sleeper Awakes
CHAPTER      : THE AWAKENING (1400 words)
READER ID    : 1363
----------------------------------------
BOOK ID      : 12163
PROJECT TITLE: Sleeper Awakes
CHAPTER      : THE PEOPLE MARCH (2000 words)
READER ID    : 2893
----------------------------------------
BOOK ID      : 12